# 00 - Setup and data

**What this notebook does**
1. Records the library versions (outputs/environment.txt).
2. Freezes the analysis plan (config/analysis_plan.json): stores it with its hash and the time in
   outputs/plan_frozen.json, *before any data is read*. Later notebooks refuse to run if the plan changes.
3. Downloads the Greek cohort (OpenNeuro ds004504 v1.0.9, 2.95 GB) if needed and verifies every file by size and
   SHA-256 against config/ds004504_v1.0.9_manifest.json. Files already present are only checked.
4. Checks the BrainLat folder: it must contain 1_AD and 5_HC, the demographic files and the .set/.fdt files.
   BrainLat cannot be downloaded automatically (access on request; see the README).

**Needs:** config/paths.yaml (copy config/paths.example.yaml and set the two data folders).

**Time:** a few seconds if the data are there; otherwise, the download.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import datetime
import json

import pandas as pd
from IPython.display import display

from eegad.config import library_versions, load_context
from eegad.data import check_brainlat, fetch_ds004504, find_brainlat_root

versions = library_versions()
for k, v in versions.items():
    print(f"{k:13s} {v}")

# versions of the original run (provenance/original_run/ambiente.txt); other versions may change the last decimals
original = {}
f_env = _root / "provenance" / "original_run" / "ambiente.txt"
if f_env.exists():
    for line in f_env.read_text(encoding="utf-8").splitlines():
        key, _, value = line.partition(":")
        original[key.strip()] = value.strip()


def same_version(key, now, orig):
    return now.split(".")[:2] == orig.split(".")[:2] if key == "python" else now == orig   # Python: major.minor


differ = {k: (versions[k], original[k]) for k in ("python", "numpy", "pandas", "scipy", "scikit-learn", "xgboost",
                                                    "mne", "matplotlib")
          if k in original and not same_version(k, versions[k], original[k])}
if differ:
    print("\nNOTE: versions different from the original run; numbers may differ slightly (notebook 08 shows how much):")
    for k, (now, orig) in differ.items():
        print(f"  {k}: {now} here, {orig} in the original run")
elif original:
    print("\nSame library versions as the original run.")
missing = [k for k in ("mne", "xgboost") if versions[k] == "MISSING"]
if "mne" in missing:
    raise ImportError("MNE is required to read the .set files: pip install -r requirements.txt")
if "xgboost" in missing:
    print("\nNOTE: xgboost is not installed; the XGBoost sensitivity analysis (notebook 06) will be skipped.")

## 1) Load the settings and freeze the analysis plan

The plan is frozen before any data file is opened. If you change config/analysis_plan.json after this, every later
notebook stops; record the change (date, what, why) in change_history and start again from here.

In [ ]:
ctx = load_context("00", require_frozen=False)
run_log = ctx.freeze_plan()
started = datetime.datetime.now().isoformat(timespec="seconds")
(ctx.out / "environment.txt").write_text(
    "\n".join(f"{k}: {v}" for k, v in versions.items()) + f"\nstarted: {started}\nquick_test_mode: {ctx.quick}\n",
    encoding="utf-8")
print(json.dumps({k: v for k, v in run_log.items() if k != "runs"}, indent=1))

## 2) Greek cohort: ds004504 v1.0.9

Downloads only what is missing or different, from the OpenNeuro public bucket (HTTPS); if that fails, it installs
openneuro-py and tries through it. Every file is checked by SHA-256. To only check (no download), set
download_ds004504: false in config/paths.yaml. Only participants.tsv and derivatives/ are needed.

In [ ]:
manifest = json.loads(ctx.ds004504_manifest.read_text(encoding="utf-8"))
assert ctx.plan["cohorts"]["greek"]["openneuro_version"] == manifest["version"], \
    "the ds004504 version in the analysis plan differs from the manifest"
record = fetch_ds004504(ctx.ds004504_dir, ctx.out / "ds004504_download.json", ctx.ds004504_manifest,
                        download=ctx.download_ds004504, base_url=ctx.ds004504_base_url)
for w in record["warnings"]:
    ctx.log.alert(w)
print(f"{record['files_ok']} of {record['files_in_manifest']} files verified by SHA-256")

## 3) BrainLat

Checks the folder structure and counts the files per group and site. A .set file whose .fdt is missing cannot be
read; those participants are reported as excluded in notebook 01 (in the original data, 14 Chilean controls).

In [ ]:
brainlat_root = find_brainlat_root(ctx.brainlat_dir, ctx.plan)
counts, demographic_files = check_brainlat(brainlat_root, ctx.plan)
display(counts)
print("Demographic files:", {k: str(v) for k, v in demographic_files.items()})
print(f"\nReady: BrainLat at {brainlat_root}; ds004504 at {ctx.ds004504_dir}. Next: notebook 01.")